# 03 Embedding Generation

This notebook is part of the **Production-Style PDF Chatbot / RAG System Project**.

## Objectives
- Understand the underlying concepts.
- Build a production-quality implementation.
- Learn the theory behind every code block.


In [1]:
# Start coding here
# ==========================================================
# Notebook 03: Embedding Generation
# ==========================================================

import pickle
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [2]:
chunks_df = pd.read_csv("data/processed/document_chunks.csv")

chunks_df.head()

,chunk_id,source_document,page_number,chunk_text,chunk_length
0,1,annual_report.pdf,1,ABBOTT INDIA LIMITED ANNUAL REPORT 2024-25 TOG...,68
1,2,annual_report.pdf,2,Abbott India Limited CONTENTS Company Overview...,500
2,3,annual_report.pdf,2,of Directors 14 Independent Auditor’s Report 1...,494
3,4,annual_report.pdf,2,34 get and stay healthy.” Munir Shaikh Wellnes...,348
4,5,annual_report.pdf,2,. Wellness for Environment 54 We cannot guaran...,470


In [3]:
MODEL_NAME = "sentence-transformers/" "all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(MODEL_NAME)

print("Model Loaded Successfully!")

'(ProtocolError('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)), '(Request ID: ea0fbf26-6f0d-4785-b99b-78ba0c66ddf5)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json
Retrying in 1s [Retry 1/5].
d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Model Loaded Successfully!


In [4]:
sample_text = "Revenue increased by 15 percent."

embedding = embedding_model.encode(sample_text)

print("Embedding Shape:", embedding.shape)

Embedding Shape: (384,)


In [5]:
print(embedding[:10])

[ 0.00908667 -0.00389833  0.04623861 -0.08636092  0.03251735 -0.03974682
 -0.01124083  0.05478781 -0.01888562  0.05156537]


In [6]:
sentence_1 = "The company's revenue increased."

sentence_2 = "The firm's sales grew."

sentence_3 = "How to bake a chocolate cake."

embeddings = embedding_model.encode([sentence_1, sentence_2, sentence_3])

print(embeddings.shape)

(3, 384)


In [7]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(embeddings)

similarity_matrix

array([[ 0.9999999 ,  0.7069137 ,  0.03591001],
       [ 0.7069137 ,  1.0000004 , -0.01619261],
       [ 0.03591001, -0.01619261,  0.99999976]], dtype=float32)

In [8]:
chunk_embeddings = embedding_model.encode(
    chunks_df["chunk_text"].tolist(), show_progress_bar=True, convert_to_numpy=True
)

Batches:   0%|          | 0/57 [00:00<?, ?it/s]

In [9]:
print("Embedding Matrix Shape:")

print(chunk_embeddings.shape)

Embedding Matrix Shape:
(1804, 384)


In [10]:
chunks_df["embedding"] = list(chunk_embeddings)

chunks_df.head()

,chunk_id,source_document,page_number,chunk_text,chunk_length,embedding
0,1,annual_report.pdf,1,ABBOTT INDIA LIMITED ANNUAL REPORT 2024-25 TOG...,68,"[0.023806352, 0.023507498, -0.024892706, 0.075..."
1,2,annual_report.pdf,2,Abbott India Limited CONTENTS Company Overview...,500,"[0.020192439, -0.03724829, -0.043878585, 0.003..."
2,3,annual_report.pdf,2,of Directors 14 Independent Auditor’s Report 1...,494,"[0.039792724, 0.058940604, -0.012694805, 0.010..."
3,4,annual_report.pdf,2,34 get and stay healthy.” Munir Shaikh Wellnes...,348,"[0.043190263, 0.063304245, -0.00028258617, 0.0..."
4,5,annual_report.pdf,2,. Wellness for Environment 54 We cannot guaran...,470,"[0.05695128, 0.07339017, 0.07628169, 0.0512070..."


In [11]:
EMBEDDING_PATH = "data/embeddings/" "chunk_embeddings.pkl"

with open(EMBEDDING_PATH, "wb") as file:

    pickle.dump(chunk_embeddings, file)

print("Embeddings saved.")

Embeddings saved.


In [12]:
chunks_df.drop(columns=["embedding"]).to_csv(
    "data/processed/document_chunks.csv", index=False
)

In [13]:
def generate_embeddings(dataframe, model):

    texts = dataframe["chunk_text"].tolist()

    embeddings = model.encode(texts, show_progress_bar=True, convert_to_numpy=True)

    return embeddings

In [14]:
embeddings = generate_embeddings(chunks_df, embedding_model)

print(embeddings.shape)

Batches:   0%|          | 0/57 [00:00<?, ?it/s]

(1804, 384)
